# 실습 1 — 프롬프트 스코어링

**같은 문제, 같은 모델, 다른 프롬프트.**
프롬프트를 고쳐서 AI의 점수를 올려보는 실습입니다.

## 진행 방법

1. **준비 단계**의 셀 2개를 순서대로 실행합니다 (셀 왼쪽 ▶ 버튼, 약 2~3분 소요)
2. **미션 1부터 4까지**, 각 미션의 셀에서 `PROMPT = """ ... """` 부분을 수정하고 다시 실행합니다
3. 점수가 오를 때까지 **수정 → 실행 → 점수 확인**을 반복합니다

> **규칙: 프롬프트 안의 `{문제}` 는 절대 지우면 안 됩니다.**
> 그 자리에 채점용 문제가 자동으로 들어갑니다. `{문제}` 위아래에 지시문을 추가하는 방식으로 수정하세요.

> 우리가 쓰는 모델(qwen2.5:0.5b)은 아주 작은 무료 모델이라 100점이 어려울 수 있습니다.
> **중요한 건 프롬프트를 고치기 전과 후의 점수 차이입니다.**

---
##  준비 1 — AI 모델 설치 (약 2분)

아래 셀을 실행하면 이 Colab 컴퓨터에 무료 로컬 AI(Ollama)가 설치됩니다.
글자가 잔뜩 올라오는 게 정상입니다. 끝날 때까지 기다리세요.

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

##  준비 2 — 모델 실행 + 다운로드 (약 1분)

AI 서버를 켜고, 우리가 사용할 작은 모델(약 400MB)을 다운로드합니다.
마지막에 **모델이 인사를 하면 준비 완료**입니다.

In [ ]:
import subprocess, time

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

!ollama pull qwen2.5:0.5b

# ---- 여기부터는 실습에 필요한 코드입니다. 수정하지 마세요! ----
import json, re, urllib.request

MODEL = "qwen2.5:0.5b"
OLLAMA_URL = "http://localhost:11434/api/chat"


def ask(prompt):
    """프롬프트를 모델에게 보내고 답변을 돌려준다."""
    body = json.dumps({
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "options": {"temperature": 0},
    }).encode("utf-8")
    req = urllib.request.Request(OLLAMA_URL, data=body,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as res:
        return json.loads(res.read().decode("utf-8"))["message"]["content"]


# 채점용 문제 세트 (수정 금지!)
TASKS = {
    "zero_shot": {
        "title": "감정 분류 (Zero-shot)",
        "labels": ["긍정", "부정", "중립"],
        "items": [
            {"input": "오늘 날씨 진짜 좋다! 소풍 가고 싶어.", "answer": "긍정"},
            {"input": "시험 망했어... 집에 가고 싶다.", "answer": "부정"},
            {"input": "오늘 점심은 김치찌개였다.", "answer": "중립"},
            {"input": "드디어 방학이다!!!", "answer": "긍정"},
            {"input": "휴대폰 액정이 깨졌다.", "answer": "부정"},
        ],
    },
    "few_shot": {
        "title": "감정 분류 - 고급 (Few-shot)",
        "labels": ["긍정", "부정", "중립"],
        "items": [
            {"input": "아 진짜 잘~한다 잘해. 또 지각이야?", "answer": "부정"},
            {"input": "이 노래 미쳤다 진짜ㅋㅋ 무한반복 중", "answer": "긍정"},
            {"input": "새로 나온 폰 실화냐? 미쳤네 개이득", "answer": "긍정"},
            {"input": "와~ 숙제가 산더미네~ 신난다~", "answer": "부정"},
            {"input": "내일 오전 9시에 수학 수업이 있다.", "answer": "중립"},
        ],
    },
    "cot": {
        "title": "수학 문제 (Chain-of-Thought)",
        "items": [
            {"input": "사탕이 15개 있다. 친구 3명에게 똑같이 나눠주면 한 명당 몇 개씩 받는가?", "answer": "5"},
            {"input": "버스에 45명이 타고 있었다. 정류장에서 12명이 내리고 8명이 탔다. 지금 버스에는 몇 명이 있는가?", "answer": "41"},
            {"input": "연필 3자루가 1500원이다. 같은 연필 12자루는 얼마인가?", "answer": "6000"},
            {"input": "지호는 매일 2000원씩 저금한다. 일주일(7일) 동안 저금하면 모두 얼마인가?", "answer": "14000"},
        ],
    },
    "role": {
        "title": "역할 부여 (Role Prompting)",
        "items": [
            {"input": "블랙홀이 뭐야?", "answer": None},
            {"input": "왜 하늘은 파란색이야?", "answer": None},
        ],
    },
}


def check(item, labels, output):
    """정답이면 True, 오답이면 False, 직접 평가 문제면 None."""
    answer = item["answer"]
    if answer is None:
        return None
    if labels:
        # 답변에서 가장 먼저 등장하는 라벨을 모델의 선택으로 본다
        found = [(output.find(l), l) for l in labels if l in output]
        return bool(found) and min(found)[1] == answer
    # 수학: 답이 독립된 숫자로 등장하는지 확인 (5가 15에 매칭되는 것 방지)
    return re.search(rf"(?<![\d.]){re.escape(answer)}(?![\d.])", output) is not None


def run_task(name, template):
    task = TASKS[name]
    template = template.strip()
    if "{문제}" not in template:
        print("[오류] 프롬프트 안에 {문제} 가 없습니다. 지우지 말고 다시 넣어주세요!")
        return
    labels = task.get("labels")

    print("=" * 60)
    print(f"  {task['title']}")
    print("=" * 60)

    results = []
    for i, item in enumerate(task["items"], 1):
        output = ask(template.replace("{문제}", item["input"])).strip()
        ok = check(item, labels, output)
        results.append(ok)
        mark = {True: "[O]", False: "[X]", None: "[?]"}[ok]
        print(f"\n{mark} 문제 {i}: {item['input']}")
        print(f"    모델 답변: {output[:300]}")
        if ok is False:
            print(f"    정답: {item['answer']}")

    scorable = [r for r in results if r is not None]
    print()
    print("-" * 60)
    if scorable:
        correct = sum(scorable)
        print(f"  점수: {correct}/{len(scorable)} ({correct / len(scorable) * 100:.0f}점)")
    else:
        print("  자동 채점 없음 — 아래 채점표를 보고 직접 채점하세요.")
    print("-" * 60)


print("모델:", ask("안녕하세요!"))
print("\n준비 완료! 아래 미션 1로 이동하세요.")

---
## 미션 1 — Zero-shot: 예시 없이 지시만으로

문장의 감정을 **긍정/부정/중립**으로 분류하는 문제입니다.
아래 셀의 처음 프롬프트는 **일부러 나쁘게** 만들어져 있습니다. 일단 그대로 실행해서 점수를 확인하고, 프롬프트를 고쳐서 점수를 올리세요.

**힌트**
- 무엇을 해야 하는지 명확하게 알려주기 (감정 분류라는 것)
- 답 형식 지정하기: "긍정, 부정, 중립 중 하나의 단어로만 답해"
- 모델이 영어나 중국어로 답하면: "한국어로만 답해" 추가

In [ ]:
# ★ 아래 따옴표 안의 프롬프트를 수정하세요 ({문제} 는 지우면 안 됨!) ★
PROMPT = """
이 문장 어때?

{문제}
"""

run_task("zero_shot", PROMPT)

---
## 미션 2 — Few-shot: 예시의 힘

이번 문제들은 **반어법, 신조어**가 섞여 있어서 지시만으로는 부족합니다.
**예시를 직접 만들어서** 프롬프트에 넣어보세요.

**힌트** — 이런 모양으로 만들면 됩니다:
```
다음 예시처럼 문장의 감정을 분류해줘. 긍정, 부정, 중립 중 하나로만 답해.

문장: "아 정말 대~단하시네요" → 부정
문장: "이 게임 개꿀잼ㅋㅋ" → 긍정
문장: "지금 밖에 비가 온다" → 중립

문장: "{문제}" →
```

In [ ]:
# ★ 예시를 추가해서 프롬프트를 완성하세요 ★
PROMPT = """
이 문장의 감정을 분류해줘.

{문제}
"""

run_task("few_shot", PROMPT)

---
## 미션 3 — Chain-of-Thought: 단계별로 생각하기

여러 단계를 거쳐야 풀리는 수학 문제입니다.
처음 프롬프트는 "답만 말해"라고 되어 있는데, 작은 모델은 이러면 계산 실수를 자주 합니다.

**힌트**
- "단계별로 차근차근 생각한 다음에 답을 말해줘" (Let's think step by step)
- 풀이 과정을 먼저 쓰고 마지막에 답을 쓰게 하기

In [ ]:
# ★ 단계별로 생각하게 만들어보세요 ★
PROMPT = """
{문제}

답만 말해.
"""

run_task("cot", PROMPT)

---
## 미션 4 — Role Prompting: 역할 부여

AI에게 **역할을 부여**하면 답변의 스타일과 품질이 달라집니다.
이 미션은 자동 채점이 없습니다. 먼저 역할 없이 실행해보고, 역할을 부여한 뒤 다시 실행해서 **전/후를 비교**하세요.

**힌트**: "너는 고등학생에게 과학을 쉽게 설명하는 선생님이야. 비유를 사용해서 3문장으로 설명해줘."

In [ ]:
# ★ 역할을 부여해보세요 ★
PROMPT = """
{문제}
"""

run_task("role", PROMPT)